In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

In [ ]:
# === 1. Tokenisation ===
def simple_tokeniser(text):
    return list(text)

def build_vocab(text):
    tokens = simple_tokeniser(text)
    vocab = sorted(set(tokens))
    stoi = {ch: i for i, ch in enumerate(vocab)}
    itos = {i: ch for ch, i in stoi.items()}
    return vocab, stoi, itos

In [ ]:
# === 2. Dataset Prep ===
def encode(text, stoi):
    return [stoi[ch] for ch in text]

def get_batches(data, block_size, batch_size):
    inputs, targets = [], []
    for _ in range(batch_size):
        i = random.randint(0, len(data) - block_size - 1)
        inputs.append(data[i:i+block_size])
        targets.append(data[i+1:i+block_size+1])
    return torch.tensor(inputs), torch.tensor(targets)

In [ ]:
# === 3. Model ===
class MiniLLM(nn.Module):
    def __init__(self, vocab_size, emb_dim=32, block_size=8, n_heads=2):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, emb_dim)
        self.position_embedding = nn.Embedding(block_size, emb_dim)
        self.attn = nn.MultiheadAttention(emb_dim, n_heads, batch_first=True)
        self.fc = nn.Linear(emb_dim, vocab_size)

    def forward(self, x):
        B, T = x.shape
        token_emb = self.token_embedding(x)                        # (B,T,E)
        pos_emb = self.position_embedding(torch.arange(T))        # (T,E)
        x = token_emb + pos_emb                                   # (B,T,E)
        attn_output, _ = self.attn(x, x, x)                        # (B,T,E)
        logits = self.fc(attn_output)                             # (B,T,V)
        return logits


In [12]:
# === 4. Training ===
def train(model, data, stoi, itos, epochs=500, block_size=8, batch_size=4, lr=1e-2):
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        x_batch, y_batch = get_batches(data, block_size, batch_size)
        logits = model(x_batch)
        loss = loss_fn(logits.view(-1, logits.size(-1)), y_batch.view(-1))

        optimiser.zero_grad()
        loss.backward()
        optimiser.step()

        if epoch % 50 == 0:
            print(f"Epoch {epoch}, Loss: {loss.item():.4f}")
            sample_text(model, stoi, itos)


In [ ]:
# === 5. Sampling ===
def sample_text(model, stoi, itos, context='H', length=50):
    model.eval()
    idxs = [stoi.get(ch, 0) for ch in context]
    for _ in range(length):
        x = torch.tensor([idxs[-8:]])  # block size
        with torch.no_grad():
            logits = model(x)
        probs = F.softmax(logits[0, -1], dim=0)
        next_idx = torch.multinomial(probs, 1).item()
        idxs.append(next_idx)
    out = ''.join([itos[i] for i in idxs])
    print("Sample:", out)
    model.train()


In [14]:
# === 6. Main Execution ===
if __name__ == "__main__":
    # Small corpus for demonstration
    corpus = "hello there general kenobi you are a bold one"
    vocab, stoi, itos = build_vocab(corpus)
    data = encode(corpus, stoi)

    model = MiniLLM(vocab_size=len(vocab))
    train(model, data, stoi, itos)

Epoch 0, Loss: 2.7747
Sample:  irrkoaheyt igonr gguauoniuyt bkrhueygdgrggr akodeu
Epoch 50, Loss: 1.0050
Sample:   loo l  lol olo loll nol ki ye u  yirolurayianoukb
Epoch 100, Loss: 0.3149
Sample:     o                                          l  o
Epoch 150, Loss: 0.0316
Sample:      l             ol  ol  ol  ol  ol  ool ol  ool 
Epoch 200, Loss: 0.0988
Sample:                                                    
Epoch 250, Loss: 0.0148
Sample:                                                    
Epoch 300, Loss: 0.6186
Sample:          l ll oll oi olo ol iloenoloonoioonooiiouoo
Epoch 350, Loss: 0.0311
Sample:                                                    
Epoch 400, Loss: 0.1617
Sample:      g gbld oberonberonbenonbibiobibiobibiobibiobib
Epoch 450, Loss: 0.1094
Sample:  ooooooooooooooooioouoouoouoouououououououououououo


In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

# === 1. Data: Use a small corpus like Shakespeare (you can replace with your own text) ===
with open("tiny_shakespeare.txt", "r", encoding="utf-8") as f:
    text = f.read()

# === 2. Tokenisation ===
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)

# === 3. Split ===
train_data = data[:int(0.9 * len(data))]
val_data = data[int(0.9 * len(data)):]

# === 4. Hyperparameters ===
block_size = 64
batch_size = 32
embedding_dim = 128
n_heads = 4
n_layers = 2
dropout = 0.1
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# === 5. Batch sampling ===
def get_batch(split):
    data_split = train_data if split == 'train' else val_data
    ix = torch.randint(len(data_split) - block_size, (batch_size,))
    x = torch.stack([data_split[i:i+block_size] for i in ix])
    y = torch.stack([data_split[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

# === 6. Transformer Components ===
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(embedding_dim, head_size, bias=False)
        self.query = nn.Linear(embedding_dim, head_size, bias=False)
        self.value = nn.Linear(embedding_dim, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) / (C ** 0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, embedding_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embedding_dim, 4 * embedding_dim),
            nn.ReLU(),
            nn.Linear(4 * embedding_dim, embedding_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        head_size = embedding_dim // n_heads
        self.sa = MultiHeadAttention(n_heads, head_size)
        self.ff = FeedForward()
        self.ln1 = nn.LayerNorm(embedding_dim)
        self.ln2 = nn.LayerNorm(embedding_dim)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

# === 7. Full Model ===
class MiniTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, embedding_dim)
        self.pos_embed = nn.Embedding(block_size, embedding_dim)
        self.blocks = nn.Sequential(*[Block() for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(embedding_dim)
        self.head = nn.Linear(embedding_dim, vocab_size)

    def forward(self, idx):
        B, T = idx.shape
        token = self.token_embed(idx)
        pos = self.pos_embed(torch.arange(T, device=idx.device))
        x = token + pos
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.head(x)
        return logits

    def generate(self, idx, max_new_tokens=100):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_id), dim=1)
        return idx

# === 8. Training ===
model = MiniTransformer().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

print("Training...")
for step in range(2000):
    x, y = get_batch('train')
    logits = model(x)
    loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 200 == 0:
        print(f"Step {step}, Loss: {loss.item():.4f}")
        context = torch.zeros((1, 1), dtype=torch.long, device=device)
        generated = model.generate(context, max_new_tokens=300)
        print(decode(generated[0].tolist()))
        print("="*40)

# Save model if needed
# torch.save(model.state_dict(), "mini_llm.pth")


Training...
Step 0, Loss: 4.3653

?RBangnhEco
awx?wtleUoLmSaHaqjeUZXKqJ'q
xqKxb$fJQczscSerqliiUAtezFlf?lrzFF!N& inkEORdFGE,aHGs'llAtXzqqqAXH,:hRAqJO
AS;gfrzReREOOySfhRcOk
tHfHum,xHEhHVuUFumaFA!CcTCsIfYudCdTl'AnX$jdVVHcmwEF$dg'OHdRRHh
fLBeOCfOOvEuS OWe.lnUDgzfCemFhLBq?WzzA$N

PzviKefuY'XCU-l- UAN-faPE?Xx-WyFlfgyr3EGTRxOYcPNsiafYJfdYW
Step 200, Loss: 2.3983

fr?
CA:
Sy thee I forlorcat.
VOLAf Cisur thil.
NTIDLAster BUE haustyo'll:
T:
F fomy tingn tighino'f ut brin aven at plo hin Ve we at hiut min woun m
In
I whag PLUmchly aitoule, in t:
Bus t my.

nd tegitstexher foraiuthy aime ind kiser cuf
Bu wio gonk
AMougveang f
RLAng PKlil rn thakrakamerer Pancheo
Step 400, Loss: 2.2074

tought Eyive wim gind epein your sil ratye ul thit jre
Meeve pitheablan late vis thing wor.

NLYORWINERIARD:
LAzRYNTIUH
Duss Firtar beach buttl be asth.
Hetingh deratle.
Mongiets Leart se thik;
Gund an the try lere, sale casuf tirt.

Dighill hacem'd sbueweaqund lout,
A thes O:
te ill hingheve and wo
Step 600, Loss:

In [18]:
# Put the model in evaluation mode
model.eval()

# Prompt to test
prompt = "to be or not"

# Encode your prompt into tokens
input_ids = torch.tensor([encode(prompt)], dtype=torch.long).to(device)

# Generate new tokens
output_ids = model.generate(input_ids, max_new_tokens=100)

# Decode and print the result
generated_text = decode(output_ids[0].tolist())
print("\nGenerated Text:\n" + generated_text)



Generated Text:
to be or not,
To your that your gall a my compeds time crow with of myseljtion!
Whereal of comes this cunkenty's


In [19]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

# === 1. Data: Use a small corpus like Shakespeare (you can replace with your own text) ===
with open("full_shakespeare.txt", "r", encoding="utf-8") as f:
    text = f.read()

# === 2. Tokenisation ===
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)

# === 3. Split ===
train_data = data[:int(0.9 * len(data))]
val_data = data[int(0.9 * len(data)):]

# === 4. Hyperparameters ===
block_size = 64
batch_size = 32
embedding_dim = 256
n_heads = 8
n_layers = 4
dropout = 0.1
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# === 5. Batch sampling ===
def get_batch(split):
    data_split = train_data if split == 'train' else val_data
    ix = torch.randint(len(data_split) - block_size, (batch_size,))
    x = torch.stack([data_split[i:i+block_size] for i in ix])
    y = torch.stack([data_split[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

# === 6. Transformer Components ===
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(embedding_dim, head_size, bias=False)
        self.query = nn.Linear(embedding_dim, head_size, bias=False)
        self.value = nn.Linear(embedding_dim, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) / (C ** 0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, embedding_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embedding_dim, 4 * embedding_dim),
            nn.ReLU(),
            nn.Linear(4 * embedding_dim, embedding_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        head_size = embedding_dim // n_heads
        self.sa = MultiHeadAttention(n_heads, head_size)
        self.ff = FeedForward()
        self.ln1 = nn.LayerNorm(embedding_dim)
        self.ln2 = nn.LayerNorm(embedding_dim)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

# === 7. Full Model ===
class MiniTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, embedding_dim)
        self.pos_embed = nn.Embedding(block_size, embedding_dim)
        self.blocks = nn.Sequential(*[Block() for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(embedding_dim)
        self.head = nn.Linear(embedding_dim, vocab_size)

    def forward(self, idx):
        B, T = idx.shape
        token = self.token_embed(idx)
        pos = self.pos_embed(torch.arange(T, device=idx.device))
        x = token + pos
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.head(x)
        return logits

    def generate(self, idx, max_new_tokens=100):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_id), dim=1)
        return idx

# === 8. Training ===
model = MiniTransformer().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

print("Training...")
for step in range(10000):
    x, y = get_batch('train')
    logits = model(x)
    loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 200 == 0:
        print(f"Step {step}, Loss: {loss.item():.4f}")
        context = torch.zeros((1, 1), dtype=torch.long, device=device)
        generated = model.generate(context, max_new_tokens=300)
        print(decode(generated[0].tolist()))
        print("="*40)

# Save model if needed
# torch.save(model.state_dict(), "mini_llm.pth")


Training...
Step 0, Loss: 4.7481
	db  b   fu )8r f“Z  o  4Bhl4VqL_” w*Zt d sSXQiÇL b, A'e] …r  c 0ms bHJ  s ot [QC A  d  us gæ ““ sigIl8Iæ *eÆbâ  x“’4EIQ  tyJHAT’I
 W4 J4yæ fU’ o8 d œ A è ht é8L Y 
  dçc3;58y æ“B ioyRfY;Qr[S1e?r’éA e Æ  qZuX bœ  w IîP foKi
Mc'iunI baoa o ly œ
Ç O lé NXÆ — E   q  e ] 
] éIœ0	j n 
X 
qn ‘  fnt u’b1  I
Step 200, Loss: 2.3142
	A.
An Grarty As gread ant,
Beeiswif the sanes.

CRIX.

Hos a hold vo fallind.maced The?

AT wee Soolched he by sthy dooth9e brut nournis hand tod the the youk
hatharn I no so.
 alithet fo dorse whe baderat Jooth,
By rot. tha dasre therodsethe penof woldentieneom
Fr busese nedssewe patand othaged.te 
Step 400, Loss: 2.0721
	atin is shoye?

FOWKIND HAFFl.
Herestilaide; to coust tway ancian.
Goun My isargarfre witonone the mangess,
Firous dearne rewannow my mame, sleake it salens shim, I nost and them,
Ando seven wo gortal.
Thy beaivin
Yentake sing a gaven, Lairtanter. FreLectyou  “I Luch.
Thaugh thavony waroons Pramings
Step 600, Loss: